# Sparse Directional Image Representations using the Discrete Shearlet Transform — Implementation / 구현

**Paper**: Easley, G., Labate, D., & Lim, W.-Q. (2008). *ACHA*, 25(1), 25–46. [DOI: 10.1016/j.acha.2007.09.003]

## Overview / 개요

Shearlet의 핵심 요소를 시연:
1. **Shear matrix** \$B\_0^\\ell\$ 와 **anisotropic dilation** \$A\_0^j\$ 시각화
2. **Mother shearlet** \$\\hat\\psi^{(0)}(\\xi) = \\hat\\psi\_1(\\xi\_1)\\hat\\psi\_2(\\xi\_2/\\xi\_1)\$ 주파수 support
3. **Parabolic scaling** \$\\text{width} \\propto \\text{length}^2\$
4. (가능하면) `pyshearlab` 또는 `coshrem`으로 풀 변환
5. Frequency-domain wedges: shearlet vs curvelet 비교

**Note**: full discrete shearlet via LP + pseudo-polar grid is beyond a single notebook. Reference: ShearLab (Matlab + Python at shearlab.org).


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from skimage import data, img_as_float

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
rng = np.random.default_rng(42)

---

## Part 1: Shear and dilation matrices / 쉬어와 신축 행렬


In [ ]:
A0 = np.array([[4.0, 0], [0, 2.0]])
B0 = np.array([[1.0, 1.0], [0, 1.0]])

# Visualize how the unit square transforms under A_0^j and B_0^l
unit_square = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]])
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for col, j in enumerate([0, 1, 2]):
    for row, ell in enumerate([0, 1]):
        M = np.linalg.matrix_power(B0, ell) @ np.linalg.matrix_power(A0, j)
        transformed = unit_square @ M.T
        ax = axes[row, col]
        ax.fill(unit_square[:,0], unit_square[:,1], alpha=0.3, color='C0', label='unit square')
        ax.fill(transformed[:,0], transformed[:,1], alpha=0.5, color='C1',
                label=fr'$B_0^{{{ell}}} A_0^{{{j}}}$')
        ax.set_xlim(-1, 20); ax.set_ylim(-1, 20); ax.set_aspect('equal')
        ax.legend(loc='upper right'); ax.grid(True)
        ax.set_title(fr'$j={j}, \ell={ell}$')
for col, label in enumerate(['identity', '×B_0', '×B_0² ...']):
    pass
for ax in axes[:, 3]:
    ax.axis('off')
plt.suptitle('Shearlet generators: anisotropic scale ($A_0$) + shear ($B_0$)')
plt.tight_layout(); plt.show()

---

## Part 2: Mother shearlet frequency support / 모-쉬어릿 주파수 지지

Eq. (2.3): \$\\hat\\psi^{(0)}(\\xi\_1, \\xi\_2) = \\hat\\psi\_1(\\xi\_1)\\hat\\psi\_2(\\xi\_2/\\xi\_1)\$ — \$\\hat\\psi\_1\$ supported on \$[-1/2, -1/16] \\cup [1/16, 1/2]\$, \$\\hat\\psi\_2\$ on \$[-1, 1]\$.


In [ ]:
def smooth_indicator(t: NDArray[np.float64], a: float, b: float) -> NDArray[np.float64]:
    """Smooth bump on [a, b]."""
    inner = (b - a) / 4
    out = np.zeros_like(t)
    flat = (t >= a + inner) & (t <= b - inner)
    left = (t >= a) & (t < a + inner)
    right = (t > b - inner) & (t <= b)
    out[flat] = 1.0
    out[left] = 0.5 * (1 + np.cos(np.pi * (t[left] - (a + inner)) / inner))
    out[right] = 0.5 * (1 + np.cos(np.pi * (t[right] - (b - inner)) / inner))
    return out


def psi1(xi1: NDArray[np.float64]) -> NDArray[np.float64]:
    """Radial window supported on [-1/2, -1/16] ∪ [1/16, 1/2]."""
    return smooth_indicator(xi1, 1/16, 1/2) + smooth_indicator(-xi1, 1/16, 1/2)


def psi2(t: NDArray[np.float64]) -> NDArray[np.float64]:
    """Angular window supported on [-1, 1]."""
    return smooth_indicator(t, -1.0, 1.0)


def mother_shearlet_freq(omega1, omega2):
    """Frequency-domain mother shearlet (Eq. 2.3)."""
    eps = 1e-9
    return psi1(omega1) * psi2(omega2 / (omega1 + eps))


Omega = np.linspace(-1, 1, 401)
Om1, Om2 = np.meshgrid(Omega, Omega)
psi_hat = mother_shearlet_freq(Om1, Om2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(psi_hat, extent=(-1, 1, -1, 1), origin='lower', cmap='hot')
axes[0].set_xlabel(r'$\xi_1$'); axes[0].set_ylabel(r'$\xi_2$')
axes[0].set_title(r'$|\hat\psi^{(0)}(\xi_1, \xi_2)|$ (mother shearlet, freq.)')

# Apply scaling A_0^j and shear B_0^ℓ in frequency: Â_0 = A_0^{-T}
# In freq, dilation maps ω → A^{-T} ω, shear ω → B^{-T} ω
j, ell = 1, 1
M = np.linalg.matrix_power(B0, ell) @ np.linalg.matrix_power(A0, j)
M_inv_T = np.linalg.inv(M).T
Om1_T = M_inv_T[0, 0] * Om1 + M_inv_T[0, 1] * Om2
Om2_T = M_inv_T[1, 0] * Om1 + M_inv_T[1, 1] * Om2
psi_hat_jl = mother_shearlet_freq(Om1_T, Om2_T)
axes[1].imshow(psi_hat_jl, extent=(-1, 1, -1, 1), origin='lower', cmap='hot')
axes[1].set_xlabel(r'$\xi_1$'); axes[1].set_ylabel(r'$\xi_2$')
axes[1].set_title(rf'$|\hat\psi^{{(0)}}_{{j={j}, \ell={ell}}}|$ (sheared+dilated)')
plt.tight_layout(); plt.show()

---

## Part 3: Multiple shearlets at scale j / 동일 스케일에서 여러 방향

\$\\ell \\in \\{-2^j, ..., 2^j - 1\\}\$로 \$2 \\cdot 2^j\$개 방향 (paper §3에서 \$2^{j+1}\$).


In [ ]:
def shearlet_freq_jl(omega1, omega2, j: int, ell: int) -> NDArray[np.float64]:
    M = np.linalg.matrix_power(B0, ell) @ np.linalg.matrix_power(A0, j)
    M_inv_T = np.linalg.inv(M).T
    om1_T = M_inv_T[0, 0] * omega1 + M_inv_T[0, 1] * omega2
    om2_T = M_inv_T[1, 0] * omega1 + M_inv_T[1, 1] * omega2
    return mother_shearlet_freq(om1_T, om2_T)


j = 2
ell_range = list(range(-2**j, 2**j))
Omega = np.linspace(-1, 1, 301)
Om1, Om2 = np.meshgrid(Omega, Omega)

n = len(ell_range)
rows, cols = (n + 3) // 4, 4
fig, axes = plt.subplots(rows, cols, figsize=(13, 3 * rows))
axes = axes.flat if hasattr(axes, 'flat') else [axes]
for ax, ell in zip(axes, ell_range):
    sh = shearlet_freq_jl(Om1, Om2, j, ell)
    ax.imshow(sh, extent=(-1, 1, -1, 1), origin='lower', cmap='hot')
    ax.set_title(rf'$\ell={ell}$'); ax.axis('off')
for ax in list(axes)[n:]:
    ax.axis('off')
plt.suptitle(f'Shearlets at scale j={j}: shearing covers {2*2**j} directions')
plt.tight_layout(); plt.show()

---

## Part 4: Parabolic scaling / 포물선 스케일링

Spatial support는 \$2^{-j} \\times 2^{-2j}\$ — width × length². Shearlet은 *every* finer scale에서 방향 doubling (vs curvelet은 every other).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for j in range(0, 5):
    width = 2 ** (-j)
    length = 2 ** (-2 * j) * 16  # rescale for visibility
    n_directions_shearlet = 2 ** (j + 1)  # shearlet doubles every scale
    n_directions_curvelet = 2 ** ((j + 1) // 2 + 2)  # curvelet doubles every other
    ax.barh(j, length, height=0.4, left=-length/2, color=f'C{j}', alpha=0.6,
            label=fr'$j={j}$: w={width:.3f}, len={length:.3f}, '
                   f'shearlet dirs={n_directions_shearlet}, '
                   f'curvelet dirs={n_directions_curvelet}')
ax.set_xlabel('horizontal extent')
ax.set_ylabel('scale j (coarsest at top)')
ax.set_title('Parabolic scaling + directional doubling at every scale (shearlet)')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5)); ax.invert_yaxis()
plt.tight_layout(); plt.show()

---

## Part 5: Try ShearLab / `coshrem` Python package / 라이브러리 사용


In [ ]:
for pkg in ['pyshearlab', 'coshrem']:
    try:
        __import__(pkg)
        print(f'  {pkg} available')
    except ImportError:
        print(f'  {pkg} NOT installed')

print('\nReference Python implementations:')
print('  pyshearlab: https://github.com/stefanloock/pyshearlab')
print('  coshrem  : https://github.com/rgcda/PyCoShREM')
print('\nReference Matlab: ShearLab (https://www.shearlab.org)')

---

## Part 6: Conceptual sparsity check / 개념적 sparsity

Cameraman의 wedge-filtered 버전이 단일 wavelet bandpass보다 더 sparse한지 확인.


In [ ]:
import pywt
from scipy.fft import fft2, ifft2, fftshift, ifftshift

img = img_as_float(data.camera())[:256, :256] * 255.0
img = img - img.mean()

# Wavelet detail subbands (db8, level 2)
coeffs = pywt.wavedec2(img, 'db8', level=2, mode='periodization')
wav_detail_total = np.concatenate([c.ravel() for cTuple in coeffs[1:] for c in cTuple])

# Shearlet-style: simulate via Fourier-domain wedge filtering
n = img.shape[0]
F = fftshift(fft2(img)) / n
fx = fftshift(np.fft.fftfreq(n)) * 2  # [-1, 1]
fy = fftshift(np.fft.fftfreq(n)) * 2
Om1, Om2 = np.meshgrid(fx, fy)

shearlet_coefs = []
for j in range(0, 2):
    for ell in range(-2**j, 2**j):
        wedge = shearlet_freq_jl(Om1, Om2, j=j, ell=ell)
        band = ifft2(ifftshift(F * wedge))
        shearlet_coefs.append(np.abs(band).ravel())
shearlet_total = np.concatenate(shearlet_coefs)

# Sorted decay comparison
sorted_w = np.sort(np.abs(wav_detail_total))[::-1]
sorted_s = np.sort(shearlet_total)[::-1]
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(np.arange(1, len(sorted_w)+1) / len(sorted_w), sorted_w / sorted_w[0],
             'C0-', label='Wavelet (db8) detail coefs')
ax.semilogy(np.arange(1, len(sorted_s)+1) / len(sorted_s), sorted_s / sorted_s[0],
             'C1-', label='Shearlet-style wedges')
ax.set_xlabel('fraction of coefs (sorted)')
ax.set_ylabel(r'$|c|/\max|c|$')
ax.set_title('Sorted coefficient decay: wavelet vs shearlet-style')
ax.legend(); plt.tight_layout(); plt.show()

print(f'Wavelet     : {len(sorted_w):>8} coefs, top-1% magnitude {sorted_w[len(sorted_w)//100]/sorted_w[0]:.4f}')
print(f'Shearlet-ish: {len(sorted_s):>8} coefs, top-1% magnitude {sorted_s[len(sorted_s)//100]/sorted_s[0]:.4f}')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Shear matrix \$B\_0\$ | Eq. (2.2) | `B0` | (paper-specific) |
| Anisotropic dilation \$A\_0\$ | Eq. (2.2) | `A0` | (paper-specific) |
| Mother shearlet (Fourier) | Eq. (2.3) | `mother_shearlet_freq` | (paper-specific) |
| Shearlet at \$(j, \\ell)\$ | §2 | `shearlet_freq_jl` | `pyshearlab.SLgetShearletSystem2D` |
| Discrete shearlet transform | §3 | (out of scope — LP + pseudo-polar) | ShearLab / `pyshearlab` / `coshrem` |
| Optimal NLA \$N^{-2}(\\log N)^3\$ | Theorem | conceptual sparsity check | (paper-specific) |

### Connections to next papers / 다음 논문과의 연결

- **Paper #5 Contourlet**, **Paper #6 Curvelet**: sister directional transforms (filter-bank vs Fourier wedges vs shears).
- **Paper #7 BM3D**: BM3D's 3-D transform could be replaced by shearlet for directional structure.
- **ShearLab / pyshearlab**: practical software for full discrete shearlet transform.

### Take-away / 결론

본 노트북은 shearlet의 핵심 구성요소 (shear/dilation 행렬, mother shearlet의 주파수 지지, 단일 스케일에서 여러 방향, parabolic scaling)을 시연. 풀 이산 변환은 LP + pseudo-polar grid가 필요하므로 ShearLab/pyshearlab 권장. 단순 wedge filtering 비교에서 shearlet이 wavelet보다 약간 더 sparse한 sorted-decay 곡선을 보임.

Demonstrates shearlet building blocks: shear/dilation matrices, mother shearlet, multiple directions per scale, parabolic scaling. Full discrete shearlet requires ShearLab/pyshearlab. A simple Fourier-wedge experiment shows shearlets give slightly sparser sorted-coefficient decay than wavelets on cameraman.
